# Rozpoznawanie odręcznie pisanych cyfr (MNIST)

Jednokierunkowa/konwolucyjna sieć neuronowa (Keras/TensorFlow), która klasyfikuje
odręcznie pisane cyfry (0-9).

MNIST to skrót od Modified National Institute of Standards and Technology.

Uruchamiaj komórki od góry do dołu. Użyj jądra **Python (handwritten-nn)**, aby
zależności projektu (tensorflow, pandas, numpy, scikit-learn, matplotlib) były
dostępne.

## Krok 1: Import bibliotek

Zanim zaczniemy, musimy zaimportować niezbędne biblioteki do manipulacji danymi, wizualizacji i budowania modelu.

In [ ]:
print("Importing modules...")
import os
# Wyciszenie informacyjnych/ostrzegawczych logów TensorFlow, takich jak poniżej:
#   WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
#   I0000 ... oneDNN custom operations are on. ... set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
# Te zmienne MUSZĄ być ustawione przed importem tensorflow, ponieważ backend C++ odczytuje je w momencie importu.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # 0=wszystko, 1=bez INFO, 2=bez WARNING, 3=bez ERROR
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # usuwa powiadomienie o niestandardowych operacjach oneDNN
# --- Podstawowe biblioteki danych / numeryczne ---
import re
import numpy as np                 # tablice numeryczne i matematyka (zmiana kształtu, argmax, normalizacja)
import pandas as pd                # wczytywanie zbioru danych CSV i przechowywanie go jako DataFrame
import matplotlib.pyplot as plt    # wykresy dokładności treningowej/walidacyjnej i wyświetlanie przykładowych cyfr
# --- Podział danych ---
from sklearn.model_selection import train_test_split  # podział danych na zbiory treningowy/walidacyjny
# --- Kontener modelu Keras ---
from tensorflow.keras.models import Sequential  # liniowy stos warstw, jedna po drugiej
# --- Elementy budulcowe warstw gęstych (w pełni połączonych) ---
from tensorflow.keras.layers import Dense, Flatten, Input
#   Input   - deklaruje kształt wejścia, np. (28, 28, 1) dla jednego obrazu w skali szarości
#   Flatten - zamienia obraz 2D/3D w wektor 1D (28x28x1 -> 784) dla warstw Dense
#   Dense   - warstwa w pełni połączona; każdy neuron łączy się ze wszystkimi wejściami (warstwy ukryte + wyjściowa)
# --- Elementy budulcowe warstw konwolucyjnych (dla CNN, lepsze dla obrazów) ---
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout
#   Conv2D       - warstwa konwolucyjna; uczy się lokalnych cech przestrzennych (krawędzie, krzywe) z obrazu 2D
#   MaxPooling2D - zmniejsza rozdzielczość map cech (zachowuje najsilniejszy sygnał, zmniejsza rozmiar, zwiększa odporność)
#   Dropout      - regularyzacja; losowo zeruje część neuronów podczas treningu, aby ograniczyć przeuczenie
# --- Przetwarzanie wstępne etykiet ---
from tensorflow.keras.utils import to_categorical  # kodowanie one-hot etykiet: 3 -> [0,0,0,1,0,0,0,0,0,0]
print("Import done.")
# Metody globalne
def shift_pixel_name(col):
    # dopasuj "pixel" po którym następuje liczba; zwiększ liczbę o 1
    m = re.fullmatch(r"pixel(\d+)", col)
    if m:
        return f"pixel{int(m.group(1)) + 1}"
    return col  # pozostaw kolumny niebędące pikselami (np. "label") bez zmian

## Krok 2: Wczytaj i zbadaj zbiór danych

Wczytamy zbiór danych i przyjrzymy się jego strukturze, aby zrozumieć cechy i etykiety.

To klasyczny zbiór danych odręcznie pisanych cyfr (0-9), powszechnie używany do trenowania i testowania modeli klasyfikacji obrazów.

Każdy obraz ma rozmiar 28x28 pikseli (784 piksele) i przedstawia jedną odręcznie pisaną cyfrę. Często nazywa się go "Hello World" uczenia maszynowego, ponieważ jest mały, dobrze poznany i szybki w treningu, co czyni go idealnym do nauki.

Zbiór danych możesz pobrać z https://media.geeksforgeeks.org/wp-content/uploads/20250407132659565012/Train.csv

Wczytuje plik CSV do DataFrame, przy czym kolumna 0 to etykieta (cyfry), a kolumny 1-784 to piksele.

In [ ]:
pd.set_option("display.max_columns", None)   # None = bez limitu
print("Loading training data from Train.csv file...")
train_data = pd.read_csv('datasets/Train.csv')
# .shape to (wiersze, kolumny) -> np. (42000, 785): 42000 obrazów x (1 etykieta + 784 piksele).
print("Done. Shape of train_data:", train_data.shape)
train_data = train_data.rename(columns=shift_pixel_name)
train_data.info()
train_data.head(5)

In [ ]:
# Podziel każdy wiersz na cechy (X) i etykietę (y). iloc to indeksowanie według pozycji całkowitej:
#   [:, 1:] -> wszystkie wiersze, kolumny 1..koniec  -> 784 wartości pikseli (wejścia)
X = train_data.iloc[:, 1:]
# X ma teraz 784 kolumny (kolumna etykiety została usunięta) -> (42000, 784)
print("Shape of X after separating features:", X.shape)
X.head(5)

## Krok 3: Przetwórz wstępnie dane

Surowe dane często wymagają oczyszczenia i sformatowania, zanim będzie można je podać do sieci neuronowej. Przetwórzmy wstępnie dane, aby przygotować je do treningu.

- Przekształć wszystkie wartości pikseli na format liczbowy i zastąp brakujące wartości zerem.
- Następnie [normalizujemy](https://www.geeksforgeeks.org/machine-learning/how-to-normalize-data-using-scikit-learn-in-python/) wartości pikseli do zakresu [0, 1], dzieląc je przez 255.0. Pomaga to modelowi uczyć się szybciej.
- Na koniec zmieniamy kształt danych, aby uwzględnić wymiar kanału, co czyni je zgodnymi z sieciami neuronowymi.

In [ ]:
# Wymuś konwersję każdej komórki na liczbę.
# errors='coerce' oznacza, że każda wartość, której nie da się przekonwertować (pusta komórka, przypadkowy tekst jak "?") stanie się NaN zamiast powodować błąd.
# Gwarantuje to, że dane są liczbowe, zanim zobaczy je model;
# wartości NaN tutaj utworzone są usuwane przez linię fillna(0) poniżej.
X = X.apply(pd.to_numeric, errors='coerce')
#  X.fillna(0) skanuje cały DataFrame i zastępuje każdy NaN zerem:
# przed:  [12, NaN, 255, NaN, 3, ...]
# po:     [12,   0, 255,   0, 3, ...]
# pojedyncza łączna liczba NaN w całym DataFrame
print ("Number of NaN is X:", X.isna().sum().sum()) # pierwszy .sum() → liczba NaN na kolumnę, drugi .sum() → suma całkowita
X = X.fillna(0)  # Ta linia zastępuje wszelkie brakujące wartości w danych pikseli zerem.
display(X.head(5))      # ładniejsze renderowanie w Jupyter

In [ ]:
# Uwaga: komórkę można wykonać tylko raz!
# / 255.0 - dzieli każdy piksel przez 255.0. Dzięki rozgłaszaniu (broadcasting) NumPy to pojedyncze dzielenie stosuje się do wszystkich wartości naraz, bez potrzeby pętli.
# Ponieważ wartości pikseli mieszczą się w zakresie 0-255, dzielenie przez 255 przeskalowuje je do zakresu 0.0-1.0:
# 0 (czarny)   -> 0.0
# 128 (szary)  -> 0.502
# 255 (biały)  -> 1.0
# Sieci neuronowe uczą się znacznie lepiej, gdy wejścia są małe i w spójnym zakresie.
X = X.values / 255.0
# Poniższa linia służy do ładniejszego renderowania w Jupyter
display(pd.DataFrame(X).head(5)) # najpierw opakuj, POTEM .head()

In [ ]:
# Uwaga: komórkę można wykonać tylko raz!
# reshape(-1, 28, 28, 1) zamienia tablicę (42000, 784) na (42000, 28, 28, 1):
#  -1  liczba obrazów                   42000
#  28  wysokość obrazu (wiersze pikseli) 28
#  28  szerokość obrazu (kolumny pikseli) 28
#   1  kanały (głębia koloru)           1 = skala szarości
# Warstwy obrazów Keras oczekują wejścia w kształcie (batch, wysokość, szerokość, kanały).
X = X.reshape(-1, 28, 28, 1)
print("Shape of X after reshaping:", X.shape)

In [ ]:
# Pokaż pierwszy wiersz
# X[0]

In [ ]:
#   [:, 0]  -> wszystkie wiersze, kolumna 0 -> etykieta cyfry 0-9 (cel)
y_orig = train_data.iloc[:, 0]
y_orig.info()

In [ ]:
counts = y_orig.value_counts().sort_index()
plt.figure(figsize=(8, 5))
bars = plt.bar(counts.index, counts.values, color="steelblue")
plt.xlabel("Digit")
plt.ylabel("Number of samples")
plt.title("Distribution of labels (0-9)")
plt.xticks(range(10))
# wypisz dokładną liczbę nad każdym słupkiem
for bar, c in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2, c, str(c),
             ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## Krok 4: Kodowanie one-hot etykiet

`y` to jednowymiarowa tablica 42000 liczb całkowitych, każda z zakresu od 0 do 9. Sieci neuronowe działają najlepiej, gdy etykiety są w formacie zwanym [kodowaniem one-hot](https://www.geeksforgeeks.org/machine-learning/ml-one-hot-encoding/). Przekonwertujmy nasze etykiety do tego formatu.

`to_categorical(y, num_classes=10)` zamienia każdą liczbę całkowitą w wektor o długości 10, który składa się z samych zer z wyjątkiem pojedynczej jedynki na pozycji odpowiadającej cyfrze:

| Oryginalna etykieta | Wektor one-hot |
|----------------|----------------|
| 0 | [1, 0, 0, 0, 0, 0, 0, 0, 0, 0] |
| 1 | [0, 1, 0, 0, 0, 0, 0, 0, 0, 0] |
| 2 | [0, 0, 1, 0, 0, 0, 0, 0, 0, 0] |
| 3 | [0, 0, 0, 1, 0, 0, 0, 0, 0, 0] |
| 4 | [0, 0, 0, 0, 1, 0, 0, 0, 0, 0] |
| 5 | [0, 0, 0, 0, 0, 1, 0, 0, 0, 0] |
| 6 | [0, 0, 0, 0, 0, 0, 1, 0, 0, 0] |
| 7 | [0, 0, 0, 0, 0, 0, 0, 1, 0, 0] |
| 8 | [0, 0, 0, 0, 0, 0, 0, 0, 1, 0] |
| 9 | [0, 0, 0, 0, 0, 0, 0, 0, 0, 1] |

Po tej linii `y` zmienia kształt z (42000,) na (42000, 10). `num_classes=10` ustala długość wektora na 10 (cyfry 0-9).

In [ ]:
print("First 5 labels:")
display(pd.DataFrame(y_orig).head(5)) # najpierw opakuj, POTEM .head()
y = to_categorical(y_orig, num_classes=10)
print("Shape of y after one-hot encoding:", y.shape)
print("First 5 one-hot encoding lables:")
display(pd.DataFrame(y).head(5)) # najpierw opakuj, POTEM .head()

## Krok 5: Podziel dane

Aby skutecznie ocenić nasz model, dzielimy dane na zbiór treningowy i zbiór walidacyjny. Tutaj używamy 80% do treningu i 20% do walidacji.

- `test_size=0.2` - odłóż 20% na walidację. Przy 42000 obrazach to podział 33600/8400.
- `random_state=42` - inicjalizuje tasowanie, dzięki czemu przy każdym uruchomieniu otrzymujesz dokładnie taki sam podział. Liczba 42 jest dowolna; dowolna stała liczba całkowita zadziała. Bez niej każde uruchomienie daje inny podział, czyniąc wyniki niereprodukowalnymi.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
# X_train obrazy do treningu:   33600 (80%)
# X_val   obrazy do walidacji:   8400 (20%)
# y_train etykiety obrazów treningowych:  33600 (80%)
# y_val   etykiety obrazów walidacyjnych:  8400 (20%)
print("X_train shape:", X_train.shape)
print("X_val   shape:", X_val.shape)

## Krok 6: Zbuduj model sieci neuronowej

Teraz zdefiniujmy architekturę. Oryginalny model gęsty (`Flatten -> Dense(128) -> Dense(64) -> Dense(10)`) spłaszcza obraz i odrzuca całą dwuwymiarową strukturę przestrzenną.

Przejście na **CNN** zachowuje siatkę 28x28 i uczy się lokalnych wzorców (krawędzie, krzywe, pętle), które są znacznie bardziej istotne dla cyfr, co przenosi MNIST z ~97-98% do zakresu 99%+. Ten notatnik używa wersji CNN:

- Warstwy `Conv2D` uczą się cech przestrzennych, `MaxPooling2D` zmniejsza rozdzielczość, `Dropout` regularyzuje.
- `Flatten` następnie zasila warstwę ukrytą `Dense(128)` i warstwę wyjściową `Dense(10, softmax)`.

Kompilujemy z optymalizatorem Adam, funkcją straty categorical cross-entropy i dokładnością jako raportowaną metryką.

In [ ]:
# Oryginalny model wyłącznie gęsty (zachowany dla odniesienia):
# model = Sequential([
#     Input(shape=(28, 28, 1)),
#     Flatten(),
#     Dense(128, activation='relu'),
#     Dense(64, activation='relu'),
#     Dense(10, activation='softmax')
# ])
model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax'),
])
# Skonfiguruj model do treningu. compile() łączy trzy rzeczy:
#   optimizer='adam'                 -> algorytm aktualizujący wagi w celu zmniejszenia błędu (adaptacyjny domyślny)
#   loss='categorical_crossentropy'  -> minimalizowana wielkość; odległość od prawdziwych etykiet one-hot
#   metrics=['accuracy']             -> raportowana (nie optymalizowana): % poprawnych predykcji
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# Wypisz tabelę warstw, kształtów wyjściowych i liczby parametrów (tylko informacyjnie).
model.summary()

## Krok 7: Trenuj model

Gdy model jest zdefiniowany, czas wytrenować go na danych treningowych. Model trenuje przez 10 epok (pełnych przejść przez zbiór treningowy), przetwarzając dane w partiach po 32 dla wydajności.

- `epochs=10` - model widzi cały zbiór treningowy 10 razy; zbyt mało powoduje niedouczenie, zbyt wiele przeuczenie.
- `batch_size=32` - wagi aktualizują się po każdej partii 32 obrazów (33600/32 = 1050 aktualizacji na epokę).
- `validation_data=(X_val, y_val)` - po każdej epoce model jest mierzony na 8400 odłożonych obrazach (nigdy nieużywanych do aktualizacji wag), dając rzetelny obraz generalizacji.

**Uwaga:** ta komórka jest tą powolną - wykonuje właściwy trening.

In [ ]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))
# --- Porównanie: odczyt metryk walidacyjnych ostatniej epoki bezpośrednio z history ---
# history.history przechowuje jedną wartość na epokę. OSTATNI wpis ([-1]) to walidacyjna
# strata/dokładność ostatniej epoki -- ten sam model/dane, które ponownie mierzy model.evaluate().
hist_val_loss = history.history['val_loss'][-1]
hist_val_accuracy = history.history['val_accuracy'][-1]
print("--- from history (last epoch, no re-computation) ---")
print(f"history val_loss:     {hist_val_loss:.4f}")
print(f"history val_accuracy: {hist_val_accuracy * 100:.2f}%")

## Krok 8: Oceń model

Gdy trening się zakończy, oceniamy wydajność na zbiorze walidacyjnym i rysujemy wykres dokładności treningowej/walidacyjnej, aby zobaczyć, jak dobrze model nauczył się w czasie (pomaga wykryć przeuczenie).

- **Dokładność (Accuracy)** - `poprawne predykcje / wszystkie predykcje`. Dokładność: 0.9724 oznacza, że 97.24% obrazów walidacyjnych zostało sklasyfikowanych poprawnie.
- **Strata (Loss)** - jak daleko przewidywane prawdopodobieństwa są od prawdy (categorical cross-entropy). Niżej znaczy lepiej; pewna siebie błędna predykcja ma bardzo wysoką stratę.

Wykres poniżej pokazuje jeden znacznik osi X na epokę.

In [ ]:
print("--- from model.evaluate (re-runs inference over X_val) ---")
val_loss, val_accuracy = model.evaluate(X_val, y_val)
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
# W notatniku wykres renderuje się w linii (bez osobnego okna).
plt.figure('Dokładność modelu per epoka')
# num_epochs jest brane z długości history, więc zawsze odpowiada liczbie
# faktycznie wytrenowanych epok. epochs = 1, 2, ... num_epochs (jedna na zapisany punkt).
num_epochs = len(history.history['accuracy'])
epochs = range(1, num_epochs + 1)
# Rysuj względem jawnych numerów epok liczonych od 1, aby oś X zaczynała się od 1 (a nie od 0).
plt.plot(epochs, history.history['accuracy'], label='Dokładność uczenia')
plt.plot(epochs, history.history['val_accuracy'], label='Dokładność walidacji/oceny')
# Jeden znacznik osi X na epokę: 1, 2, ... num_epochs.
plt.xticks(epochs)
plt.title('Dokładność modelu per epoka')
plt.xlabel('Epoka')
plt.ylabel('Dokładność')
plt.legend()
plt.show()

## Krok 9: Zapisz wytrenowany model (architektura + wagi)

Zapisany model zostanie użyty później do predykcji.

Zachowaj w pełni wytrenowany model w pojedynczym pliku `.keras`. Pakuje to wszystko, co potrzebne do ponownego użycia później: architekturę (warstwy), wyuczone wagi oraz ustawienia kompilacji (optymalizator/strata/metryki). Inne skrypty (`predict.py`, interfejsy UI) wtedy po prostu `load_model('mnist_model.keras')` - bez potrzeby ponownego treningu. Rozszerzenie `.keras` to natywny format zapisu Keras 3 oparty na zip.

In [ ]:
model.save('mnist_model.keras')
print("Saved trained model to mnist_model.keras")

## Krok 10: Wykonaj predykcje

Użyjmy wytrenowanego modelu do wykonania predykcji na nowych danych. Wczytujemy dane testowe, przetwarzamy je wstępnie tak samo jak dane treningowe i podajemy do modelu, aby uzyskać predykcje.
Testowy zbiór danych możesz pobrać z: https://media.geeksforgeeks.org/wp-content/uploads/20250707130706513705/test.zip

Pętla poniżej wyrywkowo sprawdza pierwsze 5 predykcji, rysując każdy obraz z jego przewidywaną etykietą.

In [ ]:
# Wczytaj nieoznaczony zbiór testowy (tylko piksele, bez kolumny etykiety) do DataFrame.
print("Loading training data from test.csv file...")
test_data = pd.read_csv('datasets/test.csv')
# .shape to (wiersze, kolumny) -> np. (5600, 784): 5600 obrazów x 784 piksele
print("Done. Shape of test_data:", test_data.shape)
test_data = test_data.rename(columns=shift_pixel_name)
test_data.info()
test_data.head(5)

In [ ]:
# Takie samo przetwarzanie wstępne jak przy treningu: normalizacja pikseli z 0-255 do 0.0-1.0.
# .values najpierw konwertuje DataFrame na tablicę NumPy.
X_test = test_data.values / 255.0
# Zmień kształt płaskich wierszy 784-pikselowych na (liczba_obrazów, 28, 28, 1), aby dopasować do wejścia modelu.
X_test = X_test.reshape(-1, 28, 28, 1)
# Uruchom wytrenowany model na wszystkich obrazach testowych. Każdy wiersz wyjścia to 10 prawdopodobieństw softmax.
predictions = model.predict(X_test)
# Zredukuj każdy wiersz 10 prawdopodobieństw do pojedynczej przewidywanej cyfry (indeks najwyższej wartości).
predicted_labels = np.argmax(predictions, axis=1)
# Wyrywkowo sprawdź wzrokowo pierwsze 5 predykcji: pokaż każdy obraz z jego przewidywaną etykietą.
for i in range(5):
    plt.figure()
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')  # (28,28,1) -> 2D skala szarości
    plt.title(f"Predicted: {predicted_labels[i]}")
    plt.axis('off')
    plt.show()